# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairAsim180/1_FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Unit of analysis: one row in the raw table = one content page, on one day, for one client
(report_date + client_hash_id + content_hash_id). For this contract I aggregate that up to
one row = one content page's full-month summary, over the calendar window month=2026-03 —
a mid-panel month, not the sealed _sample (June 2026) final-month test set.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip install -q duckdb

import duckdb
from google.colab import userdata

token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

# Grain check: does (report_date, client_hash_id, content_hash_id) identify one row?
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS duplicate_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  duplicate_rows
0     9841378               0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions,
scroll_events — aggregated to page-month.

Label/proxy: this raw table has no pre-built trend label. I build one within the month:
compare a page's average daily clicks in the last 7 days vs the first 7 days of March.
declining = True if last-week average < first-week average. This is a proxy, not ground
truth — one noisy month is a weak signal for real decline, but it's the only
self-contained thing this month can support without reaching into another month.

Context fields (not modeled, kept for interpretation): client_has_gsc, client_has_ga4,
gsc_data_available, ga4_data_available — explain why a row might be missing data, not
used to predict with.

Excluded: all ai_* breakdown columns (ai_chatgpt, ai_perplexity, ai_gemini, etc.) —
interesting, but mostly null this early in the panel, and not needed for a refresh-
priority signal specifically.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick look at how sparse the excluded ai_* columns actually are, to back up the "why"
sparsity_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN sessions_ai IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_ai_data
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(sparsity_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ai_data
0     9841378          6822637.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. total_impressions — knowable at the decision moment because it's already-logged GSC
   data for days that have already elapsed.
2. total_clicks — same: historical, already observed, nothing from the future.
3. avg_position — same: daily search position is logged as it happens, not predicted.
4. ctr — derived purely from already-observed impressions and clicks, no future data.
5. scroll_rate — derived purely from already-observed scroll events and pageviews.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
slice_summary = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS unique_pages,
           MIN(report_date) AS start_date,
           MAX(report_date) AS end_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE client_has_gsc IS TRUE
""").df()
print(slice_summary)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_pages start_date   end_date
0     9841378        331437 2026-03-01 2026-03-31


In [12]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE client_has_gsc IS TRUE
""").df()
print(availability)
print(f"Survival rate: {availability['gsc_available_rows'][0] / availability['total_rows'][0]:.2%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows
0     9841378           3611061.0
Survival rate: 36.69%


In [13]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(scroll_events) * 1.0 / NULLIF(SUM(ga4_pageviews), 0) AS scroll_rate
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE client_has_gsc IS TRUE AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_impressions,total_clicks,avg_position,ctr,scroll_rate
0,content_2e6360ad20fd7107,899.0,1.0,5.145765,0.001112,NaN
1,content_ac8663da7484669a,34.0,0.0,4.909314,0.000000,NaN
2,content_d49a012dcb924e31,329.0,0.0,5.177774,0.000000,NaN
3,content_614baf2af4330bd7,772.0,1.0,4.685335,0.001295,NaN
4,content_4a1ca0fa5c177e0c,14.0,0.0,4.266667,0.000000,NaN


In [14]:
label_df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN report_date <= DATE '2026-03-07' THEN gsc_clicks END) AS first_week_avg_clicks,
        AVG(CASE WHEN report_date >= DATE '2026-03-25' THEN gsc_clicks END) AS last_week_avg_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE client_has_gsc IS TRUE
    GROUP BY content_hash_id
""").df()
label_df['declining'] = label_df['last_week_avg_clicks'] < label_df['first_week_avg_clicks']

full = features.merge(label_df[['content_hash_id', 'declining', 'last_week_avg_clicks']], on='content_hash_id')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_honest = full[['total_impressions', 'total_clicks', 'avg_position', 'ctr', 'scroll_rate']].fillna(0)
y = full['declining']
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, model.predict_proba(Xte)[:,1])
print(f"Honest AUC (5 features only): {honest_auc:.3f}")

# deliberate leak: add the exact column the label is built from
X_leaked = full[['total_impressions', 'total_clicks', 'avg_position', 'ctr', 'scroll_rate', 'last_week_avg_clicks']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_leaked, y, test_size=0.3, random_state=42)
model_leak = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
leaked_auc = roc_auc_score(yte, model_leak.predict_proba(Xte)[:,1])
print(f"Leaked AUC (with last_week_avg_clicks): {leaked_auc:.3f}")

Honest AUC (5 features only): 0.776
Leaked AUC (with last_week_avg_clicks): 0.912


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- This is an unbalanced panel — client_has_gsc/client_has_ga4 vary per client, so some
  pages show zero GA4 signal for reasons unrelated to performance (the client simply
  never connected GA4), not because the page is failing.
- The label here is a within-month proxy — first-week vs last-week clicks in the same
  30-day window. It can't distinguish a real decline from ordinary week-to-week noise,
  and has no visibility into what happens after March.
- I only used month=2026-03 — one month can't separate a genuine trend from a one-off
  dip (a holiday, an algorithm update, seasonal search behavior).

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

panel_check = con.sql(f"""
    SELECT
        SUM(CASE WHEN client_has_gsc IS TRUE THEN 1 ELSE 0 END) AS has_gsc,
        SUM(CASE WHEN client_has_ga4 IS TRUE THEN 1 ELSE 0 END) AS has_ga4,
        COUNT(*) AS total
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(panel_check)
print(f"GA4 coverage: {panel_check['has_ga4'][0] / panel_check['total'][0]:.1%}")


     has_gsc    has_ga4    total
0  9841378.0  6822637.0  9841378
GA4 coverage: 69.3%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.